# Ngày 13 · Cross-validation, rò rỉ và Pipeline

- Cài k-fold và stratified k-fold có seed
- Thấy rò rỉ khi chọn đặc trưng trên toàn bộ dữ liệu trước khi chia fold
- Đặt mọi bước fit vào bên trong từng fold

**Đọc:**
- ISLR, §5.1.3 k-Fold Cross-Validation; §5.1.4 Bias-Variance Trade-Off for k-Fold
- Stanford CS229, §9.3 Model selection via cross validation
- Hastie, Tibshirani & Friedman, ESL, §7.10.2 The Wrong and Right Way to Do Cross-validation (ngoài thư mục)

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. 569 mẫu, 30 đặc trưng.
_bc = load_breast_cancer()
X_bc = _bc.data
y_bc = 1 - _bc.target   # 1 = ác tính (lớp cần phát hiện), 0 = lành tính
feature_names = list(_bc.feature_names)

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))

# Một phép chia dùng chung cho cả tuần: 70% train, 30% validation, phân tầng theo nhãn.
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
def select_top(X, y, m):
    """Chỉ số m cột có |tương quan| với y lớn nhất."""
    Xc, yc = X - X.mean(0), y - y.mean()
    corr = (Xc * yc[:, None]).sum(0) / (np.sqrt((Xc ** 2).sum(0) * (yc ** 2).sum()) + 1e-12)
    return np.argsort(-np.abs(corr))[:m]
def centroid_fit(X, y):
    return np.array([X[y == c].mean(0) for c in (0, 1)])
def centroid_predict(centers, X):
    return ((X[:, None, :] - centers[None]) ** 2).sum(2).argmin(1)
def _partition_ok(folds, n):
    vals = np.concatenate([v for _, v in folds])
    sizes = [len(v) for _, v in folds]
    return (np.array_equal(np.sort(vals), np.arange(n)) and max(sizes) - min(sizes) <= 1
            and all(np.array_equal(np.sort(np.r_[t, v]), np.arange(n)) and not set(t) & set(v) for t, v in folds))
_ys = np.r_[np.zeros(83, int), np.ones(17, int)]
def _strat_ok(fn):
    folds = fn(_ys, 5, 0)
    return _partition_ok(folds, 100) and all(abs(_ys[v].sum() - 17 / 5) <= 1 for _, v in folds)
def _noise(seed):
    r = np.random.default_rng(seed)
    return r.normal(size=(50, 1000)), np.r_[np.zeros(25, int), np.ones(25, int)]
def leaky_cv(X, y, k, m, seed):
    cols = select_top(X, y, m)   # SAI: chọn cột trên toàn bộ dữ liệu
    accs = []
    for tr, va in kfold_indices(len(y), k, seed):
        accs.append(np.mean(centroid_predict(centroid_fit(X[tr][:, cols], y[tr]), X[va][:, cols]) == y[va]))
    return float(np.mean(accs))


## k-fold và stratified k-fold

Gợi ý: kfold: hoán vị chỉ số theo seed rồi np.array_split thành k phần. Stratified: xáo trộn riêng từng lớp, nối lại theo lớp, rồi gán fold = vị trí % k để mọi fold có cỡ gần bằng nhau và cùng tỉ lệ lớp.

In [ ]:
def kfold_indices(n, k, seed):
    # Trả list k cặp (train_idx, val_idx).
    raise NotImplementedError

def stratified_kfold_indices(y, k, seed):
    raise NotImplementedError


In [ ]:
_report('d13-kfold',
    _check('k-fold phủ đủ, rời nhau, cỡ chênh ≤ 1', lambda: (_partition_ok(kfold_indices(103, 5, 0), 103), 'n = 103, k = 5.')),
    _check('k-fold có xáo trộn theo seed', lambda: (not np.array_equal(kfold_indices(50, 5, 0)[0][1], np.arange(10)) and not np.array_equal(kfold_indices(50, 5, 0)[0][1], kfold_indices(50, 5, 1)[0][1]), 'Fold đầu không phải 10 chỉ số đầu.')),
    _check('Stratified giữ tỉ lệ lớp hiếm', lambda: (_strat_ok(stratified_kfold_indices), '17 mẫu dương trên 100, mỗi fold 3–4 mẫu dương.')),
)


## Cross-validation không rò rỉ

Gợi ý: Gọi select_top bên trong vòng lặp fold, chỉ với X[train], y[train]. Dùng kfold_indices của bài trên.

In [ ]:
def honest_cv(X, y, k, m, seed):
    # Chọn m cột trong từng fold (chỉ dùng train), fit centroid, trả accuracy trung bình.
    raise NotImplementedError


In [ ]:
kfold_indices = globals().get('kfold_indices') or (lambda n, k, seed: [(np.concatenate(p[:i] + p[i+1:]), p[i]) for p in [np.array_split(np.random.default_rng(seed).permutation(n), k)] for i in range(k)])
_report('d13-honest',
    _check('Trên dữ liệu nhiễu thuần, điểm quanh mức đoán mò', lambda: (0.3 <= np.mean([honest_cv(*_noise(s), 5, 20, s) for s in range(3)]) <= 0.7, 'Nhãn độc lập với 1000 cột nhiễu; bản rò rỉ cho > 0,8.')),
    _check('Trên dữ liệu thật vẫn học được', lambda: (honest_cv(X_bc, y_bc, 5, 5, 0) > 0.85, 'Breast Cancer, 5 cột.')),
)


## Dự án

Trên Titanic, dựng Pipeline(ColumnTransformer + LogisticRegression) và so điểm cross-validation khi chuẩn hóa trong pipeline với khi chuẩn hóa toàn bộ trước. Ghi chênh lệch và giải thích.